In [1]:
# ---- Install Kaggle API ----
!pip install kaggle

from google.colab import files
# Upload your kaggle.json API token
uploaded = files.upload()

# Move kaggle.json to ~/.kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ---- Download dataset from Kaggle ----
!kaggle datasets download -d nadasalem81/ai-generated-images-vs-real-images
# Unzip into ./data folder
!unzip ai-generated-images-vs-real-images.zip -d ai-generated-images-vs-real-images

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import os
import warnings
import shutil
import numpy as np
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # Ignore broken image streams

from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# ==============================
# Ignore warnings
# ==============================
warnings.filterwarnings("ignore", message="Palette images with Transparency expressed in bytes should be converted to RGBA images")

# ==============================
# Define Real & Fake folders
# ==============================
real_dir = r"/content/ai-generated-images-vs-real-images/AI Generated Images vs Real Images/RealArt"
fake_dir = r"/content/ai-generated-images-vs-real-images/AI Generated Images vs Real Images/AiArtData"
base_train_dir = r"/content/ai-generated-images-vs-real-images/merged_data"
# ==============================
# Merge Real and Fake into one directory for flow_from_directory
# ==============================
if os.path.exists(base_train_dir):
    shutil.rmtree(base_train_dir)  # remove old merged folder
os.makedirs(os.path.join(base_train_dir, "Real"), exist_ok=True)
os.makedirs(os.path.join(base_train_dir, "Fake"), exist_ok=True)

for img in os.listdir(real_dir):
    src = os.path.join(real_dir, img)
    dst = os.path.join(base_train_dir, "Real", img)
    if os.path.isfile(src):
        shutil.copy(src, dst)

for img in os.listdir(fake_dir):
    src = os.path.join(fake_dir, img)
    dst = os.path.join(base_train_dir, "Fake", img)
    if os.path.isfile(src):
        shutil.copy(src, dst)

print("✅ Dataset folders merged successfully.")

# ==============================
# Image parameters
# ==============================
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# ==============================
# Data generators (with augmentations)
# ==============================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    horizontal_flip=True,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1
)

train_generator = train_datagen.flow_from_directory(
    base_train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    color_mode="rgb"
)

val_generator = train_datagen.flow_from_directory(
    base_train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    color_mode="rgb"
)

# ==============================
# Compute class weights
# ==============================
classes = np.unique(train_generator.classes)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_generator.classes)
class_weights = dict(zip(classes, weights))
print("⚖ Class weights:", class_weights)

# ==============================
# Build Model using Functional API
# ==============================
input_tensor = Input(shape=(224, 224, 3))
base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=input_tensor)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output_tensor = Dense(1, activation='sigmoid')(x)

# Create the model
model = Model(inputs=input_tensor, outputs=output_tensor)

# ==============================
# Compile model
# ==============================
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ==============================
# Train model
# ==============================
print("\n🚀 Starting Training...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    class_weight=class_weights,
    verbose=1
)

# ==============================
# Final Accuracies
# ==============================
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
print(f"\n✅ Final Training Accuracy: {train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {val_acc*100:.2f}%")

# ==============================
# Evaluate Model
# ==============================
y_pred_probs = model.predict(val_generator)
y_pred = (y_pred_probs > 0.5).astype(int)
y_true = val_generator.classes

print("\n=== 🧠 Final Evaluation Results ===")
print("Classification Report:\n", classification_report(y_true, y_pred, target_names=['Real', 'Fake']))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

print("\n✅ Done! Model training and evaluation completed successfully.")

# ==============================
# Save Model
# ==============================
model.save("my_model.keras")
print("\n💾 Model saved successfully as 'my_model.keras'!")

In [ ]:
import json

# Get class names from train_generator
class_indices = train_generator.class_indices

# Save class indices
with open("class_indices.json", "w") as f:
    json.dump(class_indices, f)

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import os
from tensorflow.keras.preprocessing import image

# Number of random images to display
num_samples = 5

# Get the validation folder path
val_dir = val_generator.directory

# Collect image paths and their true class labels
file_paths = []
for cls, label in val_generator.class_indices.items():
    cls_dir = os.path.join(val_dir, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            file_paths.append((os.path.join(cls_dir, fname), cls))

# Randomly pick 5 samples
samples = random.sample(file_paths, num_samples)

# Display images with predictions (vertical layout)
plt.figure(figsize=(6, num_samples * 4))  # make it taller
for i, (img_path, true_label) in enumerate(samples):
    # Load and preprocess image
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    # Predict using the model
    pred_prob = model.predict(img_array, verbose=0)[0][0]
    pred_label = "Real" if pred_prob > 0.5 else "Fake"

    # Show image and prediction results
    plt.subplot(num_samples, 1, i + 1)
    plt.imshow(image.load_img(img_path))
    plt.title(f"True: {true_label} | Pred: {pred_label} | Prob: {pred_prob:.2f}", fontsize=12)
    plt.axis("off")

plt.tight_layout()
plt.show()